In [2]:
#this involves the process of data creation
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis as PA
from modlamp.descriptors import PeptideDescriptor, GlobalDescriptor
from sklearn.model_selection import train_test_split,KFold
import pandas as pd
import os, re, math, platform
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from numpy import linalg as la
import os
import random
import warnings
warnings.filterwarnings('ignore')
import random
from fea_extract import read_fasta,insert_AAC,insert_DPC,insert_CKSAAGP,insert_CTD,insert_PAAC,insert_AAI,insert_GTPC,insert_QSO,insert_AAE,insert_PSAAC,insert_word2int,insert_ASDC
from tools import del_data
seed = 10
from pathlib import Path
Path('./results/process_data/').mkdir(exist_ok=True,parents=True)
Path('./results/balance/').mkdir(exist_ok=True,parents=True)

In [3]:
#fasta to csv format conversion
def del_data(inFile):
    seq = read_fasta(inFile)
    seqname = seq.to_numpy()
    newseq = []
    j = seqname.shape[0]
    for i in range(j):
        if 6 <= len(seqname[i][1])<=100:
            newseq.append(seqname[i])
    newseq = np.array(newseq)
    print('New Shape：', newseq.shape)
    newseq = pd.DataFrame(data=newseq, columns=["Id", "Sequence"])
    return newseq

In [5]:
from Bio import SeqIO

# Read the first FASTA file
fasta_file1 = "neg.txt"
sequences1 = list(SeqIO.parse(fasta_file1, "fasta"))

# Read the second FASTA file
fasta_file2 = "NON_ACP_GAN_300.fasta"
sequences2 = list(SeqIO.parse(fasta_file2, "fasta"))

# Combine the sequences from both files
combined_sequences = sequences1 + sequences2

# Specify the output FASTA file name
output_fasta_file = "nonacp_TRAIN.fasta"

# Write the combined sequences to the output FASTA file
with open(output_fasta_file, "w") as f:
    SeqIO.write(combined_sequences, f, "fasta")

print("Combined FASTA file created successfully.")

Combined FASTA file created successfully.


In [6]:
seq_ACP = del_data('ACP_TRAIN.fasta')
seq_non_ACP = del_data('nonacp_TRAIN.fasta')

New Shape： (1071, 2)
New Shape： (769, 2)


In [13]:
from tools import pro_data
df_train_acp = pro_data(seq_ACP)  # Modify 'pro_data' based on your data processing logic
df_train_acp.to_csv(os.path.join("data/Seq_ACP_train.csv"), index=False)

df_train_amp = pro_data(seq_non_ACP)  # Modify 'pro_data' based on your data processing logic
df_train_amp.to_csv(os.path.join("data/Seq_non_acptrain.csv"), index=False)

# If you have a combined dataset
combined_dataset = pro_data(seq_ACP + seq_non_ACP)  # Combine both datasets
df_train_combined = combined_dataset.sample(random_state=seed)
df_train_combined.to_csv(os.path.join("data/Combined_train.csv"), index=False)

print("Done!")

TypeError: expected string or bytes-like object

In [17]:
#PREPARIONG THE TRAIN AND TEST DATA FOR TRAINING
train_sets = {
    lab:pd.read_csv('data/{:s}_train.csv'.format(lab))
    for lab in ['Seq_ACP','Seq_Non_ACP']
}

train_sets['Seq_ACP'].loc[:, 'Label'] = 1 
train_sets['Seq_Non_ACP'].loc[:, 'Label'] = 0 
#test_sets['Seq_ACP'].loc[:, 'Label'] = 1
#test_sets['Seq_AMP'].loc[:, 'Label'] = 0


all_train = pd.concat([train_sets['Seq_ACP'],train_sets['Seq_Non_ACP']],ignore_index='ignore')
#all_test = pd.concat([test_sets['Seq_ACP'],test_sets['Seq_AMP']],ignore_index='ignore')
X_train = all_train.iloc[:,0:2]
#X_test = all_test.iloc[:,0:2]
y_train = all_train['Label']
#y_test = all_test["Label"]
#X_test.to_csv('data/test/X_test.csv',index = False)
#y_test.to_csv('data/test/y_test.csv',index = False)
X_train.to_csv('data/train/X_train.csv',index = False)
y_train.to_csv('data/train/y_train.csv',index = False)

In [18]:
#encdoing the peptide sequeneces
insert_list=[insert_AAC,insert_DPC,insert_CKSAAGP,insert_PSAAC,insert_CTD,insert_GTPC,
             insert_QSO,insert_AAE,insert_AAI,insert_ASDC,insert_PAAC,pro_data]
insert_str=["insert_AAC","insert_DPC","insert_CKSAAGP","insert_PSAAC","insert_CTD",
            "insert_GTPC","insert_QSO","insert_AAE","insert_AAI","insert_ASDC","insert_PAAC",'insert_All_data']

In [20]:
n = len(insert_list)
for j in range(n):
    print("encoding{}_{}".format(str(j),insert_str[j][7:]))
    #X_test = pd.read_csv('data/test/X_test.csv')
    X_train = pd.read_csv('data/train/X_train.csv')
    df_seq_train = insert_list[j](X_train)
    df_seq_train.to_csv('results/process_data/train_{}.csv'.format(insert_str[j][7:]),index =False )
    #df_seq_test = insert_list[j](X_test)
    #df_seq_test.to_csv('results/process_data/test_{}.csv'.format(insert_str[j][7:]),index =False )

encoding0_AAC
encoding1_DPC
encoding2_CKSAAGP
encoding3_PSAAC
encoding4_CTD
encoding5_GTPC
encoding6_QSO
encoding7_AAE
encoding8_AAI
encoding9_ASDC
encoding10_PAAC
encoding11_All_data


In [21]:
#wordtoint encoding for peptides
from tools import padseq
#X_test = pd.read_csv('data/test/X_test.csv')
X_train = pd.read_csv('data/train/X_train.csv')
new_seq_train = padseq(X_train)
#new_seq_test = padseq(X_test)
word2_seq_train = insert_word2int(new_seq_train)
word2_seq_train.to_csv('results/process_data/train_word2int.csv',index=False)

#word2_seq_test = insert_word2int(new_seq_test)
#word2_seq_test.to_csv('results/process_data/test_word2int.csv',index=False)

In [22]:
#encoding a balanced dataset
insert_list=[insert_AAC,insert_DPC,insert_CKSAAGP,insert_PSAAC,insert_CTD,insert_GTPC,
             insert_QSO,insert_AAE,insert_AAI,insert_ASDC,insert_PAAC,pro_data]
insert_str=["insert_AAC","insert_DPC","insert_CKSAAGP","insert_PSAAC","insert_CTD",
            "insert_GTPC","insert_QSO","insert_AAE","insert_AAI","insert_ASDC","insert_PAAC","pro_data"]